 ## Классификация

In [2]:
import os
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("/content")
cls_path = DATA_DIR / "car_evaluation.csv"
REG_PATH = DATA_DIR / "CarPrice_Assignment.csv"
col_names = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]
def load_car_evaluation(path):
    df = pd.read_csv(path, header=None, names=col_names, dtype=str)
    for c in df.columns:
        df[c] = df[c].str.strip().str.lower()
    return df

df_cls = load_car_evaluation(cls_path)
print(df_cls.head(5))
print("Shape:", df_cls.shape)
print("Target distribution:\n", df_cls['class'].value_counts())


  buying  maint doors persons lug_boot safety  class
0  vhigh  vhigh     2       2    small    low  unacc
1  vhigh  vhigh     2       2    small    med  unacc
2  vhigh  vhigh     2       2    small   high  unacc
3  vhigh  vhigh     2       2      med    low  unacc
4  vhigh  vhigh     2       2      med    med  unacc
Shape: (1728, 7)
Target distribution:
 class
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64


Можно заметить, что целевая переменная сильно не сбалансированна.
В таких условиях метрикаAccuracy (доля правильных классификаций) может быть неоптимальной, так как модель может игнорировать редкие классы. Поэтому основным показателем качества выбираем macro F1-score, усредненный F1 по всем классам.

In [3]:
X_cls = df_cls.drop(columns=['class'])
y_cls = df_cls['class']
print("Class proportions:\n", y_cls.value_counts(normalize=True))

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=RANDOM_STATE, stratify=y_cls
)
print("Train size:", X_train_cls.shape, "| Test size:", X_test_cls.shape)


Class proportions:
 class
unacc    0.700231
acc      0.222222
good     0.039931
vgood    0.037616
Name: proportion, dtype: float64
Train size: (1382, 6) | Test size: (346, 6)


Бейзлайн


In [4]:
# Преобразование: One-Hot Encoding для всех категориальных признаков
cat_cols = list(X_cls.columns)   # все столбцы категориальные
preprocess_cls = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
], remainder='drop')

# Модель KNN (baseline)
knn_cls_baseline = Pipeline(steps=[
    ('prep', preprocess_cls),
    ('knn', KNeighborsClassifier(n_neighbors=5, weights='uniform', metric='minkowski', p=2))
])
knn_cls_baseline.fit(X_train_cls, y_train_cls)
y_pred_cls = knn_cls_baseline.predict(X_test_cls)

print("=== Classification: Baseline KNN ===")
print("Accuracy:", accuracy_score(y_test_cls, y_pred_cls))
print("F1 (macro):", f1_score(y_test_cls, y_pred_cls, average='macro'))
print("\nConfusion matrix:\n", confusion_matrix(y_test_cls, y_pred_cls))
print("\nClassification report:\n", classification_report(y_test_cls, y_pred_cls, digits=4))


=== Classification: Baseline KNN ===
Accuracy: 0.8930635838150289
F1 (macro): 0.7068895341159948

Confusion matrix:
 [[ 63   0  13   1]
 [ 10   4   0   0]
 [  7   0 235   0]
 [  4   1   1   7]]

Classification report:
               precision    recall  f1-score   support

         acc     0.7500    0.8182    0.7826        77
        good     0.8000    0.2857    0.4211        14
       unacc     0.9438    0.9711    0.9572       242
       vgood     0.8750    0.5385    0.6667        13

    accuracy                         0.8931       346
   macro avg     0.8422    0.6534    0.7069       346
weighted avg     0.8923    0.8931    0.8858       346



Доминирующий класс отлично угадывается, а остальные - не очень. Попробуем улучшить бейзлайн. Основная гипотеза – использовать более подходящее кодирование категориальных признаков и настроить гиперпараметры k-NN. One-Hot Encoding может быть не оптимален для признаков, имеющих естественный порядок

In [5]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder

cv_cls = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

prep_ohe = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)])
pipe_ohe = Pipeline([('prep', prep_ohe), ('knn', KNeighborsClassifier())])
param_grid_ohe = {
    'knn__n_neighbors': [1, 3, 5, 7, 9, 15],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['hamming', 'minkowski'],
    'knn__p': [1, 2],
    'knn__algorithm': ['brute']
}

ord_maps = {
  'buying':  ['low','med','high','vhigh'],
  'maint':   ['low','med','high','vhigh'],
  'doors':   ['2','3','4','5more'],
  'persons': ['2','4','more'],
  'lug_boot':['small','med','big'],
  'safety':  ['low','med','high']
}
prep_ord = ColumnTransformer([('ord', OrdinalEncoder(categories=[ord_maps[c] for c in cat_cols]), cat_cols)])
pipe_ord = Pipeline([('prep', prep_ord), ('knn', KNeighborsClassifier())])
param_grid_ord = {
    'knn__n_neighbors': [1, 3, 5, 7, 9, 15],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['minkowski'],
    'knn__p': [1, 2]
}

results = []
for name, pipe, grid in [
    ('OHE', pipe_ohe, param_grid_ohe),
    ('Ordinal', pipe_ord, param_grid_ord)
]:
    gs = GridSearchCV(pipe, param_grid=grid, cv=cv_cls, scoring='f1_macro', n_jobs=-1, verbose=0)
    gs.fit(X_train_cls, y_train_cls)
    best_model = gs.best_estimator_
    y_pred = best_model.predict(X_test_cls)
    res = {
        'pipeline': name,
        'cv_best_f1': gs.best_score_,
        'test_acc': accuracy_score(y_test_cls, y_pred),
        'test_f1': f1_score(y_test_cls, y_pred, average='macro'),
        'best_params': gs.best_params_
    }
    results.append(res)
    print(f"\n=== {name} Encoding — best model on CV ===")
    print("Best params:", gs.best_params_)
    print(f"CV best F1_macro: {gs.best_score_:.4f}")
    print(f"Test Accuracy: {res['test_acc']:.4f}")
    print(f"Test F1_macro: {res['test_f1']:.4f}")


=== OHE Encoding — best model on CV ===
Best params: {'knn__algorithm': 'brute', 'knn__metric': 'minkowski', 'knn__n_neighbors': 9, 'knn__p': 1, 'knn__weights': 'distance'}
CV best F1_macro: 0.7335
Test Accuracy: 0.9162
Test F1_macro: 0.7454

=== Ordinal Encoding — best model on CV ===
Best params: {'knn__metric': 'minkowski', 'knn__n_neighbors': 5, 'knn__p': 2, 'knn__weights': 'distance'}
CV best F1_macro: 0.8803
Test Accuracy: 0.9509
Test F1_macro: 0.8832


В целом, порядковое кодирование признаков + оптимизированные гиперпараметры позволили значительно улучшить качество классификации по сравнению с базовым решением. Макро-F1 повысился, то есть редкие классы стали распознаваться намного лучше.

In [6]:
# Ручная проверка модели с Ordinal Encoding
enc = OrdinalEncoder(categories=[ord_maps[c] for c in X_train_cls.columns])
X_train_ord = enc.fit_transform(X_train_cls)
X_test_ord = enc.transform(X_test_cls)

knn_ord = KNeighborsClassifier(n_neighbors=7, weights='distance', metric='minkowski', p=2)
knn_ord.fit(X_train_ord, y_train_cls)
y_pred_ord = knn_ord.predict(X_test_ord)
print("Ordinal (manual) — Accuracy:", accuracy_score(y_test_cls, y_pred_ord),
      "F1_macro:", f1_score(y_test_cls, y_pred_ord, average='macro'))


Ordinal (manual) — Accuracy: 0.9710982658959537 F1_macro: 0.9284943811454803


Реализации алгоритма kNN

In [7]:
import numpy as np

class MyKNNClassifier:
    def __init__(self, n_neighbors=5, weights='uniform', p=2):
        self.n_neighbors = n_neighbors
        self.weights = weights
        self.p = p
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        return self

    def _minkowski_distances(self, x):
        d = np.abs(self.X_train - x) ** self.p
        dist = np.sum(d, axis=1) ** (1.0 / self.p)
        return dist

    def predict(self, X):
        X = np.array(X)
        y_pred = []
        for x in X:
            distances = self._minkowski_distances(x)
            neigh_idx = np.argpartition(distances, self.n_neighbors - 1)[:self.n_neighbors]
            neigh_labels = self.y_train[neigh_idx]
            neigh_dist = distances[neigh_idx]
            if self.weights == 'distance':
                if np.any(neigh_dist == 0):
                    label = neigh_labels[neigh_dist == 0][0]
                    y_pred.append(label)
                    continue
                w = 1.0 / neigh_dist
                class_weights = {}
                for lbl, wgt in zip(neigh_labels, w):
                    class_weights[lbl] = class_weights.get(lbl, 0) + wgt
                # Предсказываем класс с максимальным суммарным весом
                label = max(class_weights, key=class_weights.get)
            else:
                # При uniform взвешивании – просто выбираем самый частый класс среди соседей (majority vote)
                labels, counts = np.unique(neigh_labels, return_counts=True)
                # Если голоса поровну, np.unique возвращает отсортированные метки, выберется первая по алфавиту
                label = labels[np.argmax(counts)]
            y_pred.append(label)
        return np.array(y_pred)


In [8]:
# 1. Базовый сценарий (OHE, k=5, uniform) с MyKNNClassifier
X_train_enc = preprocess_cls.fit_transform(X_train_cls)
X_test_enc = preprocess_cls.transform(X_test_cls)
X_train_enc = X_train_enc.toarray()
X_test_enc = X_test_enc.toarray()

my_knn = MyKNNClassifier(n_neighbors=5, weights='uniform', p=2)
my_knn.fit(X_train_enc, y_train_cls)
y_pred_baseline = my_knn.predict(X_test_enc)
print("MyKNN (OHE, k=5) - Accuracy:", accuracy_score(y_test_cls, y_pred_baseline),
      "F1_macro:", f1_score(y_test_cls, y_pred_baseline, average='macro'))

# 2. Улучшенный сценарий (Ordinal, k=5, distance) с MyKNNClassifier
X_train_ord = enc.fit_transform(X_train_cls)
X_test_ord = enc.transform(X_test_cls)
my_knn2 = MyKNNClassifier(n_neighbors=5, weights='distance', p=2)
my_knn2.fit(X_train_ord, y_train_cls)
y_pred_improved = my_knn2.predict(X_test_ord)
print("MyKNN (Ordinal, k=5, dist) - Accuracy:", accuracy_score(y_test_cls, y_pred_improved),
      "F1_macro:", f1_score(y_test_cls, y_pred_improved, average='macro'))


MyKNN (OHE, k=5) - Accuracy: 0.8930635838150289 F1_macro: 0.7068895341159948
MyKNN (Ordinal, k=5, dist) - Accuracy: 0.930635838150289 F1_macro: 0.8086319477623826


# Регрессия

In [13]:
from sklearn.preprocessing import MinMaxScaler

def load_car_price_assignment(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    y = df["price"]
    X = df.drop(columns=["price", "car_ID"])
    # Выделяем бренд из имени модели
    if "CarName" in X.columns:
        carnames = X["CarName"].astype(str).str.strip().str.lower()
        X["brand"] = carnames.str.split("[ \-]", n=1, expand=True)[0]
        X = X.drop(columns=["CarName"])
    # Конвертация слов-чисел в числа (doornumber, cylindernumber)
    NUM_WORDS = {'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6, 'eight': 8, 'twelve': 12}
    for col in ["doornumber", "cylindernumber"]:
        if col in X.columns and X[col].dtype == object:
            X[col] = X[col].str.strip().str.lower().map(NUM_WORDS).astype("Int64")
    for c in X.columns:
        if X[c].dtype == object:
            X[c] = X[c].astype(str).str.strip().str.lower()
    return X, y

X_reg_raw, y_reg_raw = load_car_price_assignment(REG_PATH)
print("Regression dataset shape:", X_reg_raw.shape, "| target shape:", y_reg_raw.shape)
print("Sample columns:", list(X_reg_raw.columns)[:10], "...")
print("Target name:", y_reg_raw.name)


Regression dataset shape: (205, 24) | target shape: (205,)
Sample columns: ['symboling', 'fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'wheelbase', 'carlength', 'carwidth'] ...
Target name: price


In [14]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg_raw, y_reg_raw, test_size=0.2, random_state=42
)

cat_cols_reg = [c for c in X_train_reg.columns if X_train_reg[c].dtype == object]
num_cols_reg = [c for c in X_train_reg.columns if c not in cat_cols_reg]

for c in num_cols_reg:
    X_train_reg[c].fillna(X_train_reg[c].median(), inplace=True)
    X_test_reg[c].fillna(X_train_reg[c].median(), inplace=True)
for c in cat_cols_reg:
    mode_val = X_train_reg[c].mode().iloc[0]
    X_train_reg[c].fillna(mode_val, inplace=True)
    X_test_reg[c].fillna(mode_val, inplace=True)

# Pipeline для baseline: StandardScaler на числовые, OneHotEncoder на категориальные
preprocess_reg = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols_reg),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_reg)
])
knn_reg_baseline = Pipeline(steps=[
    ("prep", preprocess_reg),
    ("knn", KNeighborsRegressor(n_neighbors=5, weights="uniform", metric="minkowski", p=2))
])
knn_reg_baseline.fit(X_train_reg, y_train_reg)
y_pred_reg = knn_reg_baseline.predict(X_test_reg)
print("=== Regression: Baseline KNN ===")
print("MSE:", mean_squared_error(y_test_reg, y_pred_reg))
print("MAE:", mean_absolute_error(y_test_reg, y_pred_reg))
print("R^2:", r2_score(y_test_reg, y_pred_reg))


=== Regression: Baseline KNN ===
MSE: 22163071.79708022
MAE: 2763.0715365853657
R^2: 0.7192558530700428


In [15]:
X_train_reg = X_train_reg.copy()
X_test_reg = X_test_reg.copy()
# 1) Удельная мощность
X_train_reg["power_to_weight"] = X_train_reg["horsepower"] / X_train_reg["curbweight"]
X_test_reg["power_to_weight"] = X_test_reg["horsepower"] / X_test_reg["curbweight"]
# 2) Средний расход mpg
X_train_reg["mpg_mean"] = (X_train_reg["citympg"] + X_train_reg["highwaympg"]) / 2.0
X_test_reg["mpg_mean"] = (X_test_reg["citympg"] + X_test_reg["highwaympg"]) / 2.0
# 3) Плотность двигателя
X_train_reg["engine_density"] = X_train_reg["enginesize"] / X_train_reg["carlength"]
X_test_reg["engine_density"] = X_test_reg["enginesize"] / X_test_reg["carlength"]

cat_cols_reg = [c for c in X_train_reg.columns if X_train_reg[c].dtype == object]
num_cols_reg = [c for c in X_train_reg.columns if c not in cat_cols_reg]

preprocess_reg2 = ColumnTransformer(transformers=[
    ("num", MinMaxScaler(), num_cols_reg),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_reg)
])

knn_reg = Pipeline(steps=[
    ("prep", preprocess_reg2),
    ("knn", KNeighborsRegressor())
])
param_grid_reg = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2]
}
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)
gs_reg = GridSearchCV(knn_reg, param_grid=param_grid_reg, scoring='neg_mean_absolute_error', cv=cv_reg, n_jobs=-1, verbose=1)
gs_reg.fit(X_train_reg, y_train_reg)

print("=== Regression: Improved KNN + engineered features ===")
print("Best params:", gs_reg.best_params_)
y_pred_improved = gs_reg.best_estimator_.predict(X_test_reg)
print("MSE:", mean_squared_error(y_test_reg, y_pred_improved))
print("MAE:", mean_absolute_error(y_test_reg, y_pred_improved))
print("R^2:", r2_score(y_test_reg, y_pred_improved))


Fitting 5 folds for each of 24 candidates, totalling 120 fits
=== Regression: Improved KNN + engineered features ===
Best params: {'knn__n_neighbors': 5, 'knn__p': 1, 'knn__weights': 'distance'}
MSE: 9856623.09137469
MAE: 1901.7805480199463
R^2: 0.8751441466808471


In [16]:
class MyKNNRegressor:
    def __init__(self, n_neighbors=5, weights='uniform', p=2):
        self.n_neighbors = n_neighbors
        self.weights = weights
        self.p = p
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        return self

    def _minkowski_distances(self, x):
        d = np.abs(self.X_train - x) ** self.p
        dist = np.sum(d, axis=1) ** (1.0 / self.p)
        return dist

    def predict(self, X):
        X = np.array(X)
        y_pred = []
        for x in X:
            distances = self._minkowski_distances(x)
            neigh_idx = np.argpartition(distances, self.n_neighbors - 1)[:self.n_neighbors]
            neigh_values = self.y_train[neigh_idx]
            neigh_dist = distances[neigh_idx]
            if self.weights == 'distance':
                if np.any(neigh_dist == 0):
                    # Если точка совпала с обучающей, возвращаем ее значение напрямую
                    pred_val = float(neigh_values[neigh_dist == 0][0])
                    y_pred.append(pred_val)
                    continue
                w = 1.0 / neigh_dist
                # Взвешенное среднее значений соседей
                pred_val = float(np.dot(w, neigh_values) / np.sum(w))
            else:
                # Простое среднее (uniform веса)
                pred_val = float(np.mean(neigh_values))
            y_pred.append(pred_val)
        return np.array(y_pred)


In [19]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
import numpy as np

# --- БАЗОВЫЙ СЦЕНАРИЙ: StandardScaler + OHE, k=5, uniform ---
X_train_reg_base = X_train_reg.drop(columns=["power_to_weight", "mpg_mean", "engine_density"])
X_test_reg_base  = X_test_reg.drop(columns=["power_to_weight", "mpg_mean", "engine_density"])

cat_cols_reg_base = [c for c in X_train_reg_base.columns if X_train_reg_base[c].dtype == object]
num_cols_reg_base = [c for c in X_train_reg_base.columns if c not in cat_cols_reg_base]

scaler_base = StandardScaler().fit(X_train_reg_base[num_cols_reg_base])
ohe_base = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(X_train_reg_base[cat_cols_reg_base])

X_train_enc = np.hstack([
    scaler_base.transform(X_train_reg_base[num_cols_reg_base]),
    ohe_base.transform(X_train_reg_base[cat_cols_reg_base])
])
X_test_enc = np.hstack([
    scaler_base.transform(X_test_reg_base[num_cols_reg_base]),
    ohe_base.transform(X_test_reg_base[cat_cols_reg_base])
])

my_knn_reg = MyKNNRegressor(n_neighbors=5, weights='uniform', p=2)
my_knn_reg.fit(X_train_enc, y_train_reg)
y_pred_base = my_knn_reg.predict(X_test_enc)
print("MyKNNRegressor baseline - MSE:", mean_squared_error(y_test_reg, y_pred_base),
      "MAE:", mean_absolute_error(y_test_reg, y_pred_base),
      "R^2:", r2_score(y_test_reg, y_pred_base))

# --- УЛУЧШЕННЫЙ СЦЕНАРИЙ: MinMaxScaler + OHE, k=5, distance, p=1 ---
scaler_imp = MinMaxScaler().fit(X_train_reg[num_cols_reg])
ohe_imp = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(X_train_reg[cat_cols_reg])

X_train_enc2 = np.hstack([
    scaler_imp.transform(X_train_reg[num_cols_reg]),
    ohe_imp.transform(X_train_reg[cat_cols_reg])
])
X_test_enc2 = np.hstack([
    scaler_imp.transform(X_test_reg[num_cols_reg]),
    ohe_imp.transform(X_test_reg[cat_cols_reg])
])

my_knn_reg2 = MyKNNRegressor(n_neighbors=5, weights='distance', p=1)
my_knn_reg2.fit(X_train_enc2, y_train_reg)
y_pred_imp = my_knn_reg2.predict(X_test_enc2)
print("MyKNNRegressor improved - MSE:", mean_squared_error(y_test_reg, y_pred_imp),
      "MAE:", mean_absolute_error(y_test_reg, y_pred_imp),
      "R^2:", r2_score(y_test_reg, y_pred_imp))


MyKNNRegressor baseline - MSE: 22163071.79708022 MAE: 2763.0715365853657 R^2: 0.7192558530700428
MyKNNRegressor improved - MSE: 9856623.091374688 MAE: 1901.7805480199459 R^2: 0.8751441466808471
